In [1]:
import torch
import torch.nn as nn

class Planner(nn.Module):
    def __init__(self, input_channels=3, plan_dim=8):
        super().__init__()

        # ==========================================
        # STAGE 1: THE VISION MODULE
        # Squishes the image to understand its current "State"
        # ==========================================
        self.vision = nn.Sequential(
            # 256x256 -> 128x128
            nn.Conv2d(input_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # 128x128 -> 64x64
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            # 64x64 -> 32x32
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 32x32 -> 16x16
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # Global Pooling: Smashes the 16x16 grid into a single 1x1 point
            nn.AdaptiveAvgPool2d((1, 1))
        )


        # STAGE 2: THE DECISION MODULE
        # Calculates the 8-number strategy plan

        self.decision = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),

            nn.Linear(64, plan_dim),
            nn.Tanh() # Bounds the 8 numbers strictly between -1.0 and 1.0
        )

    def forward(self, current_image):
        # 1. Look at the image and extract a 256-number summary
        state_features = self.vision(current_image)

        # 2. Flatten the 3D block into a flat 1D line of numbers
        state_features = torch.flatten(state_features, 1)

        # 3. Calculate the strategy based on that summary
        plan = self.decision(state_features)

        return plan



In [2]:
#for testing
if __name__ == "__main__":
    print("--- Testing the Planner ---")
    my_planner = Planner(input_channels=3, plan_dim=8)

    # Fake current image from the environment
    dummy_state_image = torch.randn(1, 3, 256, 256)

    print(f"Input State Image: {dummy_state_image.shape}")

    # Ask the Planner for a strategy
    output_plan = my_planner(dummy_state_image)

    print(f"Output Plan Shape: {output_plan.shape}")
    print(f"Output Plan Values:\n{output_plan.detach().numpy()}")

    if output_plan.shape == (1, 8):
        print("SUCCESS! The Planner successfully generated an 8-number strategy.")

--- Testing the Planner ---
Input State Image: torch.Size([1, 3, 256, 256])
Output Plan Shape: torch.Size([1, 8])
Output Plan Values:
[[ 0.19155408  0.01030941  0.16226548  0.03216555 -0.00633972 -0.16253445
   0.00611638  0.02358814]]
SUCCESS! The Planner successfully generated an 8-number strategy.
